In [1]:
import json
import os
import cv2
import numpy as np
# from pycocotools import mask as maskUtils
import pandas as pd
from pathlib import Path 

import torch
from mmseg.apis import init_model  # Correct import for MMSegmentation 1.2.2
from mmseg.apis.inference import inference_model
import matplotlib.pyplot as plt


from dotenv import load_dotenv
# Load variables from .env file
load_dotenv()

# Get the ROOT_DIR variable
data_base_path = os.getenv("DATA_DIR")
project_root_dir = os.getenv("ROOT_DIR")

print('Project Root Path:',project_root_dir)
print('Dataset Dir:', data_base_path)

from datetime import datetime
from sklearn.preprocessing import StandardScaler
import sys
dp_path = Path(project_root_dir)/"Depth-Anything-V2"
sys.path.append(str(dp_path))
from depth_anything_v2.dpt import DepthAnythingV2


Project Root Path: /home2/pronoy.patra/Segmango_project/segmango_ssh
Dataset Dir: /scratch/janakv


xFormers not available
xFormers not available


## 1. Convert the data of image annotations into tabular data(json to csv file)

In [ ]:
# instance json file to csv conversion
def load_coco_annotations(json_path):
    """Load COCO annotations from a JSON file."""
    with open(json_path, "r") as f:
        data = json.load(f)
    return data

def create_segmentation_mask(image_shape, annotations, category_mapping):
    """
    Create a segmentation mask from COCO polygon annotations.

    Parameters:
    - image_shape: Tuple (height, width) of the image
    - annotations: List of annotations for the image
    - category_mapping: Dictionary mapping category IDs to class values

    Returns:
    - mask: NumPy array of shape (H, W) with values 0 (background), 1 (flower), 2 (fruitlet)
    """
    height, width = image_shape
    mask = np.zeros((height, width), dtype=np.uint8)  # Initialize with background (0)

    for ann in annotations:
        category_id = ann["category_id"]
        class_value = category_mapping.get(category_id, 0)  # Default to background

        if "segmentation" in ann and ann["segmentation"]:
            for seg in ann["segmentation"]:
                polygon = np.array(seg, np.int32).reshape((-1, 2))
                cv2.fillPoly(mask, [polygon], class_value)

    return mask

def get_objs_avg_size(segmented_image, class_value, min_size):
    """
    Removes objects smaller than min_size for the given class using cv2.connectedComponentsWithStats().
    """
    # Create binary mask for the given class
    binary_mask = (segmented_image == class_value).astype(np.uint8)

    # Apply connected component analysis
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)

    # Create a new mask to keep only large objects
    filtered_mask = np.zeros_like(binary_mask)

    # Iterate through components (excluding background: index 0)
    avg=0
    avg_2=0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]  # Get object size
        avg+=area
        if area >= min_size:  # Keep only large objects
            avg_2+=area
            filtered_mask[labels == i] = 1

    num_labels2, _ = cv2.connectedComponents(filtered_mask, connectivity=8)

    return num_labels, num_labels2, avg/num_labels, avg_2/num_labels2


def Get_csv_from_json(image_folder, Annotation_folder,save_file_path, prefix):

    save_mask_folder = "masks"

    category_mapping = {
        1: 1,  # Flower class → 1
        2: 2   # Fruitlet class → 2
    }

    min_size = 50  # Adjust as needed

    image_names = os.listdir(Annotation_folder)
    # prefix = 'N_03'
    json_extensions=("json")

    l_num_labels_1=[]
    l_num_labels2_1=[]
    l_avg_size_1=[]
    l_avg_size2_1=[]
    l_num_labels_2=[]
    l_num_labels2_2=[]
    l_avg_size_2=[]
    l_avg_size2_2=[]
    image_names.sort()
    image_names_n = []

    for filename in image_names:

        if filename.startswith(prefix) and filename.lower().endswith(json_extensions):
            json_path = os.path.join(Annotation_folder, filename)
            print(f"file:{filename}")
            image_names_n.append(os.path.splitext(filename)[0])

            
            os.makedirs(save_mask_folder, exist_ok=True)
            coco_data = load_coco_annotations(json_path)

            annotations_by_image = {}
            for ann in coco_data["annotations"]:
                image_id = ann["image_id"]
                if image_id not in annotations_by_image:
                    annotations_by_image[image_id] = []
                annotations_by_image[image_id].append(ann)

            # Process each image
            for image_info in coco_data["images"]:
                image_id = image_info["id"]
                image_name = image_info["file_name"]
                image_path = os.path.join(image_folder, image_name)

                # Read image to get shape
                img = cv2.imread(image_path)
                if img is None:
                    print(f"Skipping {image_name}: Image not found.")
                    continue
                
                height, width, _ = img.shape

                # Generate mask using all annotations for this image
                annotations = annotations_by_image.get(image_id, [])
                mask = create_segmentation_mask((height, width), annotations, category_mapping)
            
                num_labels_1, num_labels2_1, avg_size_1, avg_size2_1 = get_objs_avg_size(mask, class_value=1, min_size=min_size)
            
                l_num_labels_1.append(num_labels_1)
                l_num_labels2_1.append(num_labels2_1)
                l_avg_size_1.append(avg_size_1)
                l_avg_size2_1.append(avg_size2_1)
                # Filter small objeects for class 2
                num_labels_2, num_labels2_2,avg_size_2, avg_size2_2 = get_objs_avg_size(mask, class_value=2, min_size=min_size)
                
                l_num_labels_2.append(num_labels_2)
                l_num_labels2_2.append(num_labels2_2)
                l_avg_size_2.append(avg_size_2)
                l_avg_size2_2.append(avg_size2_2)

    data = { 
        'image_name_o':image_names_n,
        'n_flower_o':l_num_labels_1,
        'n_flower_t_o':l_num_labels2_1,
        'avg_flower_o':l_avg_size_1,
        'avg_flower_t_o':l_avg_size2_1,

        'n_fruit_o':l_num_labels_2,
        'n_fruit_t_o':l_num_labels2_2,
        'avg_fruit_o':l_avg_size_2,
        'avg_fruit_t_o':l_avg_size2_2
    }

    df = pd.DataFrame(data)

    save_file_path.mkdir(parents=True, exist_ok=True)

    # Save as CSV
    csv_file = Path(save_file_path)/f"visit_{prefix}_instance_json.csv"
    
    # csv_file = "visit_01_data_instance_json.csv"
    df.to_csv(csv_file, index=False)

    print(f"CSV file saved: {csv_file}")
    # Set paths and category mappings

# image_folder = Path(data_base_path)/'Dataset_images_2025'
# Annotation_folder = Path(data_base_path)/'Dataset_annotations_2025'
# save_file_path = Path(project_root_dir)/'data/tabular_data'
# prefix = 'N_03'
# Get_csv_from_json(image_folder, Annotation_folder, save_file_path, prefix)



In [ ]:
# will be used as ground truth(output)
for i in ['02','03','04','05','06','07','08']:
    prefix = i
    image_folder = Path(data_base_path)/'Dataset_images_2024'
    Annotation_folder = Path(data_base_path)/'Dataset_annotations_2024'
    save_file_path = Path(project_root_dir)/'data/tabular_data'

    Get_csv_from_json(image_folder, Annotation_folder, save_file_path, prefix)

for i in ['N_02','N_03']:
    prefix = i
    image_folder = Path(data_base_path)/'Dataset_images_2025'
    Annotation_folder = Path(data_base_path)/'Dataset_annotations_2025'
    save_file_path = Path(project_root_dir)/'data/tabular_data'

    Get_csv_from_json(image_folder, Annotation_folder, save_file_path, prefix)


file:02_01_01.json
file:02_01_02.json
file:02_01_03.json
file:02_01_04.json
file:02_01_05.json
file:02_01_06.json
file:02_01_07.json
file:02_01_08.json
file:02_02_01.json
file:02_02_02.json
file:02_02_03.json
file:02_02_04.json
file:02_02_05.json
file:02_02_06.json
file:02_02_07.json
file:02_02_08.json
file:02_03_01.json
file:02_03_02.json
file:02_03_03.json
file:02_03_04.json
file:02_03_05.json
file:02_03_06.json
file:02_03_07.json
file:02_03_08.json
file:02_04_01.json
file:02_04_02.json
file:02_04_03.json
file:02_04_04.json
file:02_04_05.json
file:02_04_06.json
file:02_04_07.json
file:02_04_08.json
file:02_05_01.json
file:02_05_02.json
file:02_05_03.json
file:02_05_04.json
file:02_05_05.json
file:02_05_06.json
file:02_05_07.json
file:02_05_08.json
file:02_06_01.json
file:02_06_02.json
file:02_06_03.json
file:02_06_04.json
file:02_06_05.json
file:02_06_06.json
file:02_06_07.json
file:02_06_08.json
file:02_07_01.json
file:02_07_02.json
file:02_07_03.json
file:02_07_04.json
file:02_07_0

## 2. generete the image annotations and then conver them in csv

In [7]:
# segmentation model mask file to csv conversion
# gdown --fuzzy --folder https://drive.google.com/drive/folders/1LUz7nOI3Q7iv6TVqz9l9k7ng5SW8xJz7?usp=sharing

# Load the model
config_file = Path(project_root_dir)/'data/Model_weights/approach-1/segformer/mango_sense_segformer_10.py'  # Update with your config file path (files available)
checkpoint_file = Path(project_root_dir)/'data/Model_weights/approach-1/segformer/iter_49160.pth'  # Update with your checkpoint file path
model = init_model(str(config_file), str(checkpoint_file), device='cuda' if torch.cuda.is_available() else 'cpu')

def divide_image_into_patches(image, patch_size=1000):
    """Splits an image into non-overlapping patches of size patch_size x patch_size."""
    h, w, _ = image.shape
    patches = []
    positions = []
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patch = image[i:i+patch_size, j:j+patch_size]
            patches.append(patch)
            positions.append((i, j))
    return patches, positions

def process_patches(patches, model):
    """Runs segmentation on each patch."""
    segmented_patches = []
    for patch in patches:
        result = inference_model(model, patch)
        # print(result.pred_sem_seg.data.cpu())
        # plt.imshow(result.pred_sem_seg.data.cpu().numpy()[0])
        mask = result.pred_sem_seg.data.cpu().numpy()[0]  # Extract the first result if multiple
        segmented_patches.append(mask)
    return segmented_patches

def merge_patches(segmented_patches, positions, original_size=(8000, 6000), patch_size=1000):
    """Reconstructs the full-size segmentation mask from patches."""
    full_mask = np.zeros(original_size, dtype=np.uint8)
    for patch, (i, j) in zip(segmented_patches, positions):
        full_mask[i:i+patch_size, j:j+patch_size] = patch
    return full_mask

def get_objs_avg_size(segmented_image, class_value, min_size):
    """
    Removes objects smaller than min_size for the given class using cv2.connectedComponentsWithStats().
    """
    # Create binary mask for the given class
    binary_mask = (segmented_image == class_value).astype(np.uint8)

    # Apply connected component analysis
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)

    # Create a new mask to keep only large objects
    filtered_mask = np.zeros_like(binary_mask)

    # Iterate through components (excluding background: index 0)
    avg=0
    avg_2=0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]  # Get object size
        avg+=area
        if area >= min_size:  # Keep only large objects
            avg_2+=area
            filtered_mask[labels == i] = 1

    num_labels2, _ = cv2.connectedComponents(filtered_mask, connectivity=8)

    return num_labels, num_labels2, avg/num_labels, avg_2/num_labels2
# Load and process the image

def Get_csv_from_seg_model(image_folder, prefix, save_file_path):


    min_size = 50  # Adjust as needed
    # folder_path = "/scratch/janakv/New_mangosense_images"
    image_names = os.listdir(image_folder)
    # prefix = 'N_01'
    image_extensions=("jpg", "png", "jpeg", "bmp", "tiff")
    # print(image_names)
    l_num_labels_1=[]
    l_num_labels2_1=[]
    l_avg_size_1=[]
    l_avg_size2_1=[]
    l_num_labels_2=[]
    l_num_labels2_2=[]
    l_avg_size_2=[]
    l_avg_size2_2=[]
    image_names.sort()
    image_names_n = []
    # new_prefix = '08'
    for filename in image_names:
        # print(f"file: {filename}")
        # if filename.startswith(prefix):
        if filename.startswith(prefix) and filename.lower().endswith(image_extensions):
            image_path = os.path.join(image_folder, filename)
            # new_filename = new_prefix + filename[len(prefix):]
            # image_path_2 = os.path.join(folder_path, new_filename)
            # image_path = '01_05_01.jpg'  # Update with your image path
            print(f"file:{filename}")
            image_names_n.append(os.path.splitext(filename)[0])
            image = cv2.imread(image_path)
            patches, positions = divide_image_into_patches(image)

            # Segment each patch
            segmented_patches = process_patches(patches, model)

            # Merge patches back into the full image
            segmentation_mask = merge_patches(segmented_patches, positions)

            # Save the segmentation result
            # cv2.imwrite('segmentation_result.png', segmentation_mask)

        # Set minimum object size

        # Filter small objects for class 1
            num_labels_1, num_labels2_1, avg_size_1, avg_size2_1 = get_objs_avg_size(segmentation_mask, class_value=1, min_size=min_size)
            # print(f"Number of objects for class 1 (OpenCV):{num_labels}",f"after removal of small obj:{num_labels2}")
            # print(f"Class1 Avg size: {avg_size}")
            l_num_labels_1.append(num_labels_1)
            l_num_labels2_1.append(num_labels2_1)
            l_avg_size_1.append(avg_size_1)
            l_avg_size2_1.append(avg_size2_1)
            # Filter small objeects for class 2
            num_labels_2, num_labels2_2,avg_size_2, avg_size2_2 = get_objs_avg_size(segmentation_mask, class_value=2, min_size=min_size)
            # print(f"Number of objects for class 1 (OpenCV):{num_labels}",f"after removal of small obj:{num_labels2}")
            # print(f"Class2 Avg size: {avg_size2}")
            l_num_labels_2.append(num_labels_2)
            l_num_labels2_2.append(num_labels2_2)
            l_avg_size_2.append(avg_size_2)
            l_avg_size2_2.append(avg_size2_2)

    data = { 
        'image_name':image_names_n,
        'n_flower':l_num_labels_1,
        'n_flower_t':l_num_labels2_1,
        'avg_flower':l_avg_size_1,
        'avg_flower_t':l_avg_size2_1,

        'n_fruit':l_num_labels_2,
        'n_fruit_t':l_num_labels2_2,
        'avg_fruit':l_avg_size_2,
        'avg_fruit_t':l_avg_size2_2
    }

    df = pd.DataFrame(data)

    # Save as CSV
    save_file_path.mkdir(parents=True, exist_ok=True)

    csv_file = save_file_path/f"visit_{prefix}_data.csv"
    df.to_csv(csv_file, index=False)

    print(f"CSV file saved: {csv_file}")

/home2/pronoy.patra/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


Loads checkpoint by local backend from path: /home2/pronoy.patra/Segmango_project/segmango_ssh/data/Model_weights/approach-1/segformer/iter_49160.pth


In [8]:
prefix = '08'
image_folder = Path(data_base_path)/'Dataset_images_2024'
Annotation_folder = Path(data_base_path)/'Dataset_annotations_2024'
save_file_path = Path(project_root_dir)/'data/tabular_data'
Get_csv_from_seg_model(image_folder,prefix, save_file_path)


file:08_01_01.jpg
file:08_01_02.jpg
file:08_01_03.jpg
file:08_01_04.jpg
file:08_01_05.jpg
file:08_01_06.jpg
file:08_01_07.jpg
file:08_01_08.jpg
file:08_02_01.jpg
file:08_02_02.jpg
file:08_02_03.jpg
file:08_02_04.jpg
file:08_02_05.jpg
file:08_02_06.jpg
file:08_02_07.jpg
file:08_02_08.jpg
file:08_03_01.jpg
file:08_03_02.jpg
file:08_03_03.jpg
file:08_03_04.jpg
file:08_03_05.jpg
file:08_03_06.jpg
file:08_03_07.jpg
file:08_03_08.jpg
file:08_04_01.jpg
file:08_04_02.jpg
file:08_04_03.jpg
file:08_04_04.jpg
file:08_04_05.jpg
file:08_04_06.jpg
file:08_04_07.jpg
file:08_04_08.jpg
file:08_05_01.jpg
file:08_05_02.jpg
file:08_05_03.jpg
file:08_05_04.jpg
file:08_05_05.jpg
file:08_05_06.jpg
file:08_05_07.jpg
file:08_05_08.jpg
file:08_06_01.jpg
file:08_06_02.jpg
file:08_06_03.jpg
file:08_06_04.jpg
file:08_06_05.jpg
file:08_06_06.jpg
file:08_06_07.jpg
file:08_06_08.jpg
file:08_07_01.jpg
file:08_07_02.jpg
file:08_07_03.jpg
file:08_07_04.jpg
file:08_07_05.jpg
file:08_07_06.jpg
file:08_07_07.jpg
file:08_07

In [ ]:
# will be use as input features
for i in ['01','02','03','04','05','06','07', '08']:
    prefix = i
    image_folder = Path(data_base_path)/'Dataset_images_2024'
    Annotation_folder = Path(data_base_path)/'Dataset_annotations_2024'
    save_file_path = Path(project_root_dir)/'data/tabular_data'
    Get_csv_from_seg_model(image_folder,prefix, save_file_path)

for i in ['N_01','N_02','N_03']:
# for i in ['N_03']:
    prefix = i
    image_folder = Path(data_base_path)/'Dataset_images_2025'
    Annotation_folder = Path(data_base_path)/'Dataset_annotations_2025'
    save_file_path = Path(project_root_dir)/'data/tabular_data'
    Get_csv_from_seg_model(image_folder,prefix, save_file_path)

## 3. Prepare the csv with combination of image data, weather data, image scaling factors
Now, the weather data is preproceed and image scaling based on ralative scaling generated using DepthAnything model. Both new modality data is combined with the image data.
Here we used the DepthAnything model weights directly without fine-tunning.

In [2]:
data_w1  =pd.read_csv(f'{project_root_dir}/data/2024_feb_march_april_weather_data.csv')
len(data_w1)
data_w1.head()

,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,sealevelpressure,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
0,"17.241127227100648, 78.42922916230114",2024-02-01 00:00:00,21.0,21.0,15.0,68.57,0.0,0,NaN,0,...,1017.0,0.0,6.0,0.0,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"
1,"17.241127227100648, 78.42922916230114",2024-02-01 01:00:00,20.0,20.0,15.0,72.94,0.0,0,NaN,0,...,1017.0,0.0,6.0,0.0,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"
2,"17.241127227100648, 78.42922916230114",2024-02-01 02:00:00,20.0,20.0,15.0,72.95,0.0,0,NaN,0,...,1016.0,0.0,6.0,0.0,0.0,0,NaN,Clear,clear-night,"43128599999,43128099999,VOHS"
3,"17.241127227100648, 78.42922916230114",2024-02-01 03:00:00,19.0,19.0,15.0,77.61,0.0,0,NaN,0,...,1016.0,0.0,6.0,0.0,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"
4,"17.241127227100648, 78.42922916230114",2024-02-01 04:00:00,19.0,19.0,16.0,82.75,0.0,0,NaN,0,...,1016.0,0.0,6.0,0.0,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"


In [3]:
columns_to_scale = ['temp', 'feelslike','dew','humidity','precip','precipprob',  'sealevelpressure', 'cloudcover', 
                    'visibility', 'solarradiation', 'windgust', 'windspeed']

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(data_w1[columns_to_scale]), columns=columns_to_scale, index=data_w1.index)

data_w1.update(df_scaled)
data_w1.head()

,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,sealevelpressure,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
0,"17.241127227100648, 78.42922916230114",2024-02-01 00:00:00,-1.536070,-1.586181,-0.239204,1.041956,-0.062871,-0.08077,NaN,0,...,0.834974,-1.916428,0.045148,-0.754133,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"
1,"17.241127227100648, 78.42922916230114",2024-02-01 01:00:00,-1.728377,-1.784342,-0.239204,1.273961,-0.062871,-0.08077,NaN,0,...,0.834974,-1.916428,0.045148,-0.754133,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"
2,"17.241127227100648, 78.42922916230114",2024-02-01 02:00:00,-1.728377,-1.784342,-0.239204,1.274492,-0.062871,-0.08077,NaN,0,...,0.535242,-1.916428,0.045148,-0.754133,0.0,0,NaN,Clear,clear-night,"43128599999,43128099999,VOHS"
3,"17.241127227100648, 78.42922916230114",2024-02-01 03:00:00,-1.920685,-1.982503,-0.239204,1.521892,-0.062871,-0.08077,NaN,0,...,0.535242,-1.916428,0.045148,-0.754133,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"
4,"17.241127227100648, 78.42922916230114",2024-02-01 04:00:00,-1.920685,-1.982503,0.061122,1.794776,-0.062871,-0.08077,NaN,0,...,0.535242,-1.916428,0.045148,-0.754133,0.0,0,NaN,Clear,clear-night,"43128599999,VOHS"


In [ ]:
# 2024 data
csv_file = f"{project_root_dir}/data/mlp_all_data_with_time_weather_scale_treewise_2024.csv"
images_path= f'{data_base_path}/Dataset_images_2024'
model_checkpoints_path = f"{project_root_dir}/data/Model_weights/approach-1/depth_anything_v2/depth_anything_v2_vits.pth"

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(model_checkpoints_path, map_location='cpu'))
model = model.to(DEVICE).eval()


def safe_percentile_ratio(depth, c_i, percentile, epsilon=0.01):
    left = depth[:, :c_i]
    right = depth[:, c_i:]

    p_left = np.percentile(left, percentile)
    p_right = np.percentile(right, percentile)

    if abs(p_right) < epsilon:
        return p_left / epsilon  # or np.nan
    else:
        return p_left / p_right



# Example: access r_10
# print("r_10 =", ratios["r_10"])



def prepare_scale_factors(data_1,data_2, images_path):
    image_names = data_1['image_name']
    # raw_img_1 = cv2.imread(os.path.join(images_path, '01_01_01' + '.jpg'))
    r_sums = []
    r_maxs = []
    r_stds = []
    img_names = []
    r_10s,r_20s, r_30s, r_40s, r_50s, r_60s, r_70s, r_80s, r_90s = [],[],[],[],[],[],[],[],[]
    for i_n in image_names:
        raw_img_1 = cv2.imread(os.path.join(images_path, i_n[:-2] + '01.jpg'))  #1st image of each tree

        raw_img_2 = cv2.imread(os.path.join(images_path, i_n + '.jpg'))
        # remove the 15% area to remove extra objects from the image
        r_s_1, r_e_1, c_s_1 ,c_e_1 = int(raw_img_1.shape[0]*0.15),int(raw_img_1.shape[0]*0.85),int(raw_img_1.shape[1]*0.15),int(raw_img_1.shape[1]*0.85)
        r_s_2, r_e_2, c_s_2 ,c_e_2 = int(raw_img_2.shape[0]*0.15),int(raw_img_2.shape[0]*0.85),int(raw_img_2.shape[1]*0.15),int(raw_img_2.shape[1]*0.85)
        img_c = np.concatenate((raw_img_1[r_s_1:r_e_1,c_s_1:c_e_1], raw_img_2[r_s_2:r_e_2,c_s_2:c_e_2]),axis=1)
        depth = model.infer_image(img_c) # HxW raw depth map in numpy

        c_i = c_e_1-c_s_1

        r_sum = np.sum(depth[:,:c_i])/np.sum(depth[:,c_i:])
        r_max = np.max(depth[:,:c_i])/np.max(depth[:,c_i:])
        r_std = np.std(depth[:,:c_i])/np.std(depth[:,c_i:])


        percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
        ratios = {}

        for p in percentiles:
            ratios[f"r_{p}"] = safe_percentile_ratio(depth, c_i, p)

        r_sums.append(r_sum)
        r_maxs.append(r_max)
        r_stds.append(r_std)
    
        r_10s.append(ratios["r_10"])
        r_20s.append(ratios["r_20"])
        r_30s.append(ratios["r_30"])
        r_40s.append(ratios["r_40"])
        r_50s.append(ratios["r_50"])
        r_60s.append(ratios["r_60"])
        r_70s.append(ratios["r_70"])
        r_80s.append(ratios["r_80"])
        r_90s.append(ratios["r_90"])
        # print('yes')
    # scale_f = pd.DataFrame()
    # scale_f['image_name'] = img_names
    data_1['scale_sum_r'] = r_sums
    data_1['scale_max_r'] = r_maxs
    data_1['scale_std_r'] = r_stds
    data_1['scale_p_10'] = r_10s
    data_1['scale_p_20'] = r_20s
    data_1['scale_p_30'] = r_30s
    data_1['scale_p_40'] = r_40s
    data_1['scale_p_50'] = r_50s
    data_1['scale_p_60'] = r_60s
    data_1['scale_p_70'] = r_70s
    data_1['scale_p_80'] = r_80s
    data_1['scale_p_90'] = r_90s

    image_names = data_2['image_name_o']
    r_sums = []
    r_maxs = []
    r_stds = []
    img_names = []
    r_10s,r_20s, r_30s, r_40s, r_50s, r_60s, r_70s, r_80s, r_90s = [],[],[],[],[],[],[],[],[]

    for i_n in image_names:
        raw_img_1 = cv2.imread(os.path.join(images_path, i_n[:-2] + '01.jpg'))  #1st image of each tree

        raw_img_2 = cv2.imread(os.path.join(images_path, i_n + '.jpg'))
        # remove the 15% area to remove extra objects from the image
        r_s_1, r_e_1, c_s_1 ,c_e_1 = int(raw_img_1.shape[0]*0.15),int(raw_img_1.shape[0]*0.85),int(raw_img_1.shape[1]*0.15),int(raw_img_1.shape[1]*0.85)
        r_s_2, r_e_2, c_s_2 ,c_e_2 = int(raw_img_2.shape[0]*0.15),int(raw_img_2.shape[0]*0.85),int(raw_img_2.shape[1]*0.15),int(raw_img_2.shape[1]*0.85)
        img_c = np.concatenate((raw_img_1[r_s_1:r_e_1,c_s_1:c_e_1], raw_img_2[r_s_2:r_e_2,c_s_2:c_e_2]),axis=1)
        depth = model.infer_image(img_c) # HxW raw depth map in numpy

        c_i = c_e_1-c_s_1

        r_sum = np.sum(depth[:,:c_i])/np.sum(depth[:,c_i:])
        r_max = np.max(depth[:,:c_i])/np.max(depth[:,c_i:])
        r_std = np.std(depth[:,:c_i])/np.std(depth[:,c_i:])


        percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
        ratios = {}

        for p in percentiles:
            ratios[f"r_{p}"] = safe_percentile_ratio(depth, c_i, p)

        r_sums.append(r_sum)
        r_maxs.append(r_max)
        r_stds.append(r_std)

        r_10s.append(ratios["r_10"])
        r_20s.append(ratios["r_20"])
        r_30s.append(ratios["r_30"])
        r_40s.append(ratios["r_40"])
        r_50s.append(ratios["r_50"])
        r_60s.append(ratios["r_60"])
        r_70s.append(ratios["r_70"])
        r_80s.append(ratios["r_80"])
        r_90s.append(ratios["r_90"])

    data_2['scale_sum_r_o'] = r_sums
    data_2['scale_max_r_o'] = r_maxs
    data_2['scale_std_r_o'] = r_stds
    data_2['scale_p_10_o'] = r_10s
    data_2['scale_p_20_o'] = r_20s
    data_2['scale_p_30_o'] = r_30s
    data_2['scale_p_40_o'] = r_40s
    data_2['scale_p_50_o'] = r_50s
    data_2['scale_p_60_o'] = r_60s
    data_2['scale_p_70_o'] = r_70s
    data_2['scale_p_80_o'] = r_80s
    data_2['scale_p_90_o'] = r_90s


    data_2['avg_flower_o_s'] = data_2.avg_flower_o*data_2.scale_sum_r_o
    data_2['avg_fruit_o_s'] = data_2.avg_fruit_o*data_2.scale_sum_r_o
    print('yes:prepare_scale_factors')
    return data_1,data_2





def prepare_weather_data(filtered_df,images_path,tree_n):
    
    # filtered_df
    filtered_df['preciptype'] = filtered_df['preciptype'].replace('rain', 1)

    filtered_df.fillna(0, inplace=True)

    temp = filtered_df['temp'].mean()
    dew = filtered_df['dew'].mean()
    precip = filtered_df['precip'].mean()
    precipprob = filtered_df['precipprob'].mean()
    sealevelpressure = filtered_df['sealevelpressure'].mean()
    cloudcover = filtered_df['cloudcover'].mean()
    visibility = filtered_df['visibility'].mean()
    solarradiation = filtered_df['solarradiation'].mean()
    severerisk = filtered_df['severerisk'].mean()
    preciptype = filtered_df['preciptype'].mean()
    num_rows = 96
    # 'temp','dew','humidity','precip','precipprob',  'sealevelpressure', 'cloudcover', 'visibility', 'solarradiation',
    # Create a DataFrame with columns having fixed values
    new_df = pd.DataFrame({
        'temp': [temp] * num_rows,    # All values are 0
        'dew': [dew] * num_rows,     # All values are 10
        'precip': [precip] * num_rows,      
        'precipprob': [precipprob] * num_rows ,    
        'visibility': [visibility] * num_rows  ,    
        'solarradiation': [solarradiation] * num_rows  ,     
    # 'severerisk', 'preciptype'['rain'], 
        'severerisk': [severerisk] * num_rows   ,
        'preciptype': [preciptype] * num_rows   
    })

    # 'windgust', 'windspeed', 'winddir'
    bins = [0, 22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5, 360]
    labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW', 'N*']

    # Use pd.cut to categorize wind directions
    # Note: N appears twice (0-22.5 and 337.5-360), so we handle the wraparound
    filtered_df['wind_direction']  = pd.cut(filtered_df['winddir'], bins=bins, labels=labels, include_lowest=True, right=False)
    count = filtered_df['wind_direction'].value_counts()
    # print(count)
    count['N'] = count['N']+count['N*']

    count = count.drop('N*',axis=0)
    count_n = [count[n] for n in ['SW','S','SE','E','NE','N','NW','W']]
    count_n = (count_n/sum(count_n)).tolist()
    repeated_values = count_n * tree_n


    # Merge N and N* counts
    filtered_df.loc[:, 'wind_direction'] = filtered_df['wind_direction'].replace('N*', 'N')

    # Compute mean windgust and windspeed for each direction
    mean_values = filtered_df.groupby('wind_direction')[['windgust', 'windspeed']].mean()
    mean_values

    windgust_n = [mean_values['windgust'][n] for n in ['SW','S','SE','E','NE','N','NW','W']]
    windgust_n = (windgust_n/sum(windgust_n)).tolist()

    windspeed_n = [mean_values['windspeed'][n] for n in ['SW','S','SE','E','NE','N','NW','W']]
    windspeed_n = (windspeed_n/sum(windspeed_n)).tolist()

    new_df['winddir'] = repeated_values
    new_df['windgust']= windgust_n*tree_n
    new_df['windspeed']= windspeed_n*tree_n
    return new_df

data_1_all = []
data_2_all = []
for i in range(1,9):
    data_1_  =pd.read_csv(f"{project_root_dir}/data/tabular_data/visit_0{i}_data.csv")
    if i!=1:
        data_2_ = pd.read_csv(f"{project_root_dir}/data/tabular_data/visit_0{i}_instance_json.csv")
    else:
        data_2_ = pd.read_csv(f"{project_root_dir}/data/tabular_data/visit_0{i+1}_instance_json.csv")
    data_1_,data_2_ = prepare_scale_factors(data_1_,data_2_,images_path)
    data_1_all.append(data_1_)
    data_2_all.append(data_2_)

dates = [[2024,2,16],[2024,2,28],[2024,3,8],[2024,3,19],[2024,4,2],[2024,4,16],[2024,4,23],[2024,4,30]]
dates_n = pd.to_datetime(["-".join(map(str, date)) for date in dates])
all_data = []
weather_data = []
tree_n=12
for i in range(1,8):
    for j in range(2,9):
        if i<j:
            # data_1  =pd.read_csv(f'visit_0{i}_data.csv')
            # data_2 = pd.read_csv(f"visit_0{j}_data_instance_json.csv")
            # data_1,data_2 = prepare_scale_factors(data_1,data_2)
            data_1 = data_1_all[i-1]
            data_2 = data_2_all[j-1]
            # scale_sum_r	scale_max_r	scale_std_r
            data_ = pd.concat([data_1['image_name'],data_2['image_name_o'],data_1['n_flower'],data_1['avg_flower'],data_1['n_fruit'],data_1['avg_fruit'],
            data_1['scale_sum_r'],data_1['scale_max_r'],data_1['scale_std_r'],
            data_1['scale_p_10'], data_1['scale_p_20'], data_1['scale_p_30'], data_1['scale_p_40'], data_1['scale_p_50'],
            data_1['scale_p_60'], data_1['scale_p_70'], data_1['scale_p_80'], data_1['scale_p_90'],
            data_2['n_flower_o'],data_2['avg_flower_o'],data_2['n_fruit_o'],data_2['avg_fruit_o'], 
            data_2['scale_sum_r_o'], data_2['scale_max_r_o'],data_2['scale_std_r_o'],
            data_2['scale_p_10_o'], data_2['scale_p_20_o'], data_2['scale_p_30_o'], data_2['scale_p_40_o'], data_2['scale_p_50_o'],
            data_2['scale_p_60_o'], data_2['scale_p_70_o'], data_2['scale_p_80_o'], data_2['scale_p_90_o'],
            data_2['avg_flower_o_s'], data_2['avg_fruit_o_s']], axis=1, ignore_index=False
            )
            
            date1 = datetime(dates[i-1][0],dates[i-1][1],dates[i-1][2])
            date2 = datetime(dates[j-1][0],dates[j-1][1],dates[j-1][2])
            data_['time'] = (date2 - date1).days

                    
            start_date = str(dates_n[i-1])
            end_date = str(dates_n[j-1])
            # # Filter the dataframe based on the date range
            filtered_df = data_w1[(data_w1["datetime"] >= start_date) & (data_w1["datetime"] <= end_date)]
            # weather_data.append(filtered_df)
            new_df = prepare_weather_data(filtered_df, images_path, tree_n)
            combined_df = pd.concat([data_, new_df], axis=1)
            all_data.append(combined_df)
            print('done:',i,j)
            # display(data_)
            # break
        else:
            print('not',i,j)


data_f = all_data[0]
for i in range(1,len(all_data)):
    data_f = pd.concat([data_f, all_data[i]], ignore_index=True)
# len(data_f), data_f

data_f.to_csv(csv_file, index=False)

In [11]:
data_w1  =pd.read_csv(f'{project_root_dir}/data/2025_feb_march_april_weather_data.csv')

len(data_w1)
data_w1.head()
columns_to_scale = ['temp', 'feelslike','dew','humidity','precip','precipprob',  'sealevelpressure', 'cloudcover', 
                    'visibility', 'solarradiation', 'windgust', 'windspeed']

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(data_w1[columns_to_scale]), columns=columns_to_scale, index=data_w1.index)

data_w1.update(df_scaled)
data_w1.head()

,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,sealevelpressure,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
0,"17.430288917800137, 78.31955488810969",2025-02-01T00:00:00,-1.462777,-1.485591,0.023858,0.990196,-0.042237,-0.088182,NaN,0,...,0.118128,0.028860,-0.419478,-0.777254,0.0,0,NaN,Partially cloudy,partly-cloudy-night,"43128599999,VOHS"
1,"17.430288917800137, 78.31955488810969",2025-02-01T01:00:00,-1.563383,-1.589705,0.023858,1.100971,-0.042237,-0.088182,NaN,0,...,0.118128,0.028860,-0.419478,-0.777254,0.0,0,NaN,Partially cloudy,partly-cloudy-night,"43128599999,VOHS"
2,"17.430288917800137, 78.31955488810969",2025-02-01T02:00:00,-1.764595,-1.797932,0.023858,1.326372,-0.042237,-0.088182,NaN,0,...,0.118128,0.028860,-0.419478,-0.777254,0.0,0,NaN,Partially cloudy,partly-cloudy-night,"43128599999,VOHS"
3,"17.430288917800137, 78.31955488810969",2025-02-01T03:00:00,-1.965808,-2.006160,0.023858,1.567668,-0.042237,-0.088182,NaN,0,...,-0.219530,-0.082000,-0.419478,-0.777254,0.0,0,NaN,Partially cloudy,partly-cloudy-night,VOHS
4,"17.430288917800137, 78.31955488810969",2025-02-01T04:00:00,-1.945687,-1.985337,-0.180856,1.319148,-0.042237,-0.088182,NaN,0,...,-0.219530,0.131193,-0.419478,-0.777254,0.0,0,NaN,Partially cloudy,partly-cloudy-night,43128599999


In [12]:
# 2025 data
csv_file = f"{project_root_dir}/data/mlp_all_data_with_time_weather_scale_treewise_2025.csv"
images_path= f'{data_base_path}/Dataset_images_2025'
model_checkpoints_path = f"{project_root_dir}/data/Model_weights/approach-1/depth_anything_v2/depth_anything_v2_vits.pth"


sys.path.append(os.path.abspath('/home2/janakv/yield_pred/Depth-Anything-V2')) 
from depth_anything_v2.dpt import DepthAnythingV2

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(model_checkpoints_path, map_location='cpu'))
model = model.to(DEVICE).eval()


def safe_percentile_ratio(depth, c_i, percentile, epsilon=0.01):
    left = depth[:, :c_i]
    right = depth[:, c_i:]

    p_left = np.percentile(left, percentile)
    p_right = np.percentile(right, percentile)

    if abs(p_right) < epsilon:
        return p_left / epsilon  # or np.nan
    else:
        return p_left / p_right



# Example: access r_10
# print("r_10 =", ratios["r_10"])



def prepare_scale_factors(data_1,data_2, images_path):
    image_names = data_1['image_name']
    # raw_img_1 = cv2.imread(os.path.join(images_path, '01_01_01' + '.jpg'))
    r_sums = []
    r_maxs = []
    r_stds = []
    img_names = []
    r_10s,r_20s, r_30s, r_40s, r_50s, r_60s, r_70s, r_80s, r_90s = [],[],[],[],[],[],[],[],[]
    for i_n in image_names:
        raw_img_1 = cv2.imread(os.path.join(images_path, i_n[:-2] + '01.jpg'))  #1st image of each tree

        raw_img_2 = cv2.imread(os.path.join(images_path, i_n + '.jpg'))
        # remove the 15% area to remove extra objects from the image
        r_s_1, r_e_1, c_s_1 ,c_e_1 = int(raw_img_1.shape[0]*0.15),int(raw_img_1.shape[0]*0.85),int(raw_img_1.shape[1]*0.15),int(raw_img_1.shape[1]*0.85)
        r_s_2, r_e_2, c_s_2 ,c_e_2 = int(raw_img_2.shape[0]*0.15),int(raw_img_2.shape[0]*0.85),int(raw_img_2.shape[1]*0.15),int(raw_img_2.shape[1]*0.85)
        img_c = np.concatenate((raw_img_1[r_s_1:r_e_1,c_s_1:c_e_1], raw_img_2[r_s_2:r_e_2,c_s_2:c_e_2]),axis=1)
        depth = model.infer_image(img_c) # HxW raw depth map in numpy

        c_i = c_e_1-c_s_1

        r_sum = np.sum(depth[:,:c_i])/np.sum(depth[:,c_i:])
        r_max = np.max(depth[:,:c_i])/np.max(depth[:,c_i:])
        r_std = np.std(depth[:,:c_i])/np.std(depth[:,c_i:])


        percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
        ratios = {}

        for p in percentiles:
            ratios[f"r_{p}"] = safe_percentile_ratio(depth, c_i, p)

        r_sums.append(r_sum)
        r_maxs.append(r_max)
        r_stds.append(r_std)
    
        r_10s.append(ratios["r_10"])
        r_20s.append(ratios["r_20"])
        r_30s.append(ratios["r_30"])
        r_40s.append(ratios["r_40"])
        r_50s.append(ratios["r_50"])
        r_60s.append(ratios["r_60"])
        r_70s.append(ratios["r_70"])
        r_80s.append(ratios["r_80"])
        r_90s.append(ratios["r_90"])

    data_1['scale_sum_r'] = r_sums
    data_1['scale_max_r'] = r_maxs
    data_1['scale_std_r'] = r_stds
    data_1['scale_p_10'] = r_10s
    data_1['scale_p_20'] = r_20s
    data_1['scale_p_30'] = r_30s
    data_1['scale_p_40'] = r_40s
    data_1['scale_p_50'] = r_50s
    data_1['scale_p_60'] = r_60s
    data_1['scale_p_70'] = r_70s
    data_1['scale_p_80'] = r_80s
    data_1['scale_p_90'] = r_90s

    image_names = data_2['image_name_o']
    r_sums = []
    r_maxs = []
    r_stds = []
    img_names = []
    r_10s,r_20s, r_30s, r_40s, r_50s, r_60s, r_70s, r_80s, r_90s = [],[],[],[],[],[],[],[],[]

    for i_n in image_names:
        raw_img_1 = cv2.imread(os.path.join(images_path, i_n[:-2] + '01.jpg'))  #1st image of each tree

        raw_img_2 = cv2.imread(os.path.join(images_path, i_n + '.jpg'))
        # remove the 15% area to remove extra objects from the image
        r_s_1, r_e_1, c_s_1 ,c_e_1 = int(raw_img_1.shape[0]*0.15),int(raw_img_1.shape[0]*0.85),int(raw_img_1.shape[1]*0.15),int(raw_img_1.shape[1]*0.85)
        r_s_2, r_e_2, c_s_2 ,c_e_2 = int(raw_img_2.shape[0]*0.15),int(raw_img_2.shape[0]*0.85),int(raw_img_2.shape[1]*0.15),int(raw_img_2.shape[1]*0.85)
        img_c = np.concatenate((raw_img_1[r_s_1:r_e_1,c_s_1:c_e_1], raw_img_2[r_s_2:r_e_2,c_s_2:c_e_2]),axis=1)
        depth = model.infer_image(img_c) # HxW raw depth map in numpy

        c_i = c_e_1-c_s_1

        r_sum = np.sum(depth[:,:c_i])/np.sum(depth[:,c_i:])
        r_max = np.max(depth[:,:c_i])/np.max(depth[:,c_i:])
        r_std = np.std(depth[:,:c_i])/np.std(depth[:,c_i:])


        percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
        ratios = {}

        for p in percentiles:
            ratios[f"r_{p}"] = safe_percentile_ratio(depth, c_i, p)

        r_sums.append(r_sum)
        r_maxs.append(r_max)
        r_stds.append(r_std)

        r_10s.append(ratios["r_10"])
        r_20s.append(ratios["r_20"])
        r_30s.append(ratios["r_30"])
        r_40s.append(ratios["r_40"])
        r_50s.append(ratios["r_50"])
        r_60s.append(ratios["r_60"])
        r_70s.append(ratios["r_70"])
        r_80s.append(ratios["r_80"])
        r_90s.append(ratios["r_90"])

    data_2['scale_sum_r_o'] = r_sums
    data_2['scale_max_r_o'] = r_maxs
    data_2['scale_std_r_o'] = r_stds
    data_2['scale_p_10_o'] = r_10s
    data_2['scale_p_20_o'] = r_20s
    data_2['scale_p_30_o'] = r_30s
    data_2['scale_p_40_o'] = r_40s
    data_2['scale_p_50_o'] = r_50s
    data_2['scale_p_60_o'] = r_60s
    data_2['scale_p_70_o'] = r_70s
    data_2['scale_p_80_o'] = r_80s
    data_2['scale_p_90_o'] = r_90s


    data_2['avg_flower_o_s'] = data_2.avg_flower_o*data_2.scale_sum_r_o
    data_2['avg_fruit_o_s'] = data_2.avg_fruit_o*data_2.scale_sum_r_o
    print('yes:prepare_scale_factors')
    return data_1,data_2





def prepare_weather_data(filtered_df,images_path, tree_n=12):
    
    # filtered_df
    filtered_df['preciptype'] = filtered_df['preciptype'].replace('rain', 1)

    filtered_df.fillna(0, inplace=True)

    temp = filtered_df['temp'].mean()
    dew = filtered_df['dew'].mean()
    precip = filtered_df['precip'].mean()
    precipprob = filtered_df['precipprob'].mean()
    sealevelpressure = filtered_df['sealevelpressure'].mean()
    cloudcover = filtered_df['cloudcover'].mean()
    visibility = filtered_df['visibility'].mean()
    solarradiation = filtered_df['solarradiation'].mean()
    severerisk = filtered_df['severerisk'].mean()
    preciptype = filtered_df['preciptype'].mean()
    num_rows = tree_n*8
    # 'temp','dew','humidity','precip','precipprob',  'sealevelpressure', 'cloudcover', 'visibility', 'solarradiation',
    # Create a DataFrame with columns having fixed values
    new_df = pd.DataFrame({
        'temp': [temp] * num_rows,    # All values are 0
        'dew': [dew] * num_rows,     # All values are 10
        'precip': [precip] * num_rows,      
        'precipprob': [precipprob] * num_rows ,    
        'visibility': [visibility] * num_rows  ,    
        'solarradiation': [solarradiation] * num_rows  ,     
    # 'severerisk', 'preciptype'['rain'], 
        'severerisk': [severerisk] * num_rows   ,
        'preciptype': [preciptype] * num_rows   
    })

    # 'windgust', 'windspeed', 'winddir'
    bins = [0, 22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5, 360]
    labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW', 'N*']

    # Use pd.cut to categorize wind directions
    # Note: N appears twice (0-22.5 and 337.5-360), so we handle the wraparound
    filtered_df['wind_direction']  = pd.cut(filtered_df['winddir'], bins=bins, labels=labels, include_lowest=True, right=False)
    count = filtered_df['wind_direction'].value_counts()
    # print(count)
    count['N'] = count['N']+count['N*']

    count = count.drop('N*',axis=0)
    count_n = [count[n] for n in ['SW','S','SE','E','NE','N','NW','W']]
    count_n = (count_n/sum(count_n)).tolist()
    repeated_values = count_n * tree_n

    # Merge N and N* counts
    filtered_df.loc[:, 'wind_direction'] = filtered_df['wind_direction'].replace('N*', 'N')

    # Compute mean windgust and windspeed for each direction
    mean_values = filtered_df.groupby('wind_direction')[['windgust', 'windspeed']].mean()
    mean_values

    windgust_n = [mean_values['windgust'][n] for n in ['SW','S','SE','E','NE','N','NW','W']]
    windgust_n = (windgust_n/sum(windgust_n)).tolist()

    windspeed_n = [mean_values['windspeed'][n] for n in ['SW','S','SE','E','NE','N','NW','W']]
    windspeed_n = (windspeed_n/sum(windspeed_n)).tolist()

    new_df['winddir'] = repeated_values
    new_df['windgust']= windgust_n*tree_n
    new_df['windspeed']= windspeed_n*tree_n
    return new_df

data_1_all = []
data_2_all = []

for i in range(1,4):
    data_1_  =pd.read_csv(f"{project_root_dir}/data/tabular_data/visit_N_0{i}_data.csv")
    if i!=1:
        data_2_ = pd.read_csv(f"{project_root_dir}/data/tabular_data/visit_N_0{i}_instance_json.csv")
    else:
        data_2_ = pd.read_csv(f"{project_root_dir}/data/tabular_data/visit_N_0{i+1}_instance_json.csv")
    data_1_,data_2_ = prepare_scale_factors(data_1_,data_2_,images_path)
    data_1_all.append(data_1_)
    data_2_all.append(data_2_)

# dates = [[2024,2,16],[2024,2,28],[2024,3,8],[2024,3,19],[2024,4,2],[2024,4,16],[2024,4,23],[2024,4,30]]
dates = [[2025,2,20],[2025,3,28],[2025,4,21]]
dates_n = pd.to_datetime(["-".join(map(str, date)) for date in dates])
all_data = []
weather_data = []
tree_n=20
for i in range(1,3):
    for j in range(2,4):
        if i<j:
            # data_1  =pd.read_csv(f'visit_0{i}_data.csv')
            # data_2 = pd.read_csv(f"visit_0{j}_data_instance_json.csv")
            # data_1,data_2 = prepare_scale_factors(data_1,data_2)
            data_1 = data_1_all[i-1]
            data_2 = data_2_all[j-1]
            # scale_sum_r	scale_max_r	scale_std_r
            data_ = pd.concat([data_1['image_name'],data_2['image_name_o'],data_1['n_flower'],data_1['avg_flower'],data_1['n_fruit'],data_1['avg_fruit'],
            data_1['scale_sum_r'],data_1['scale_max_r'],data_1['scale_std_r'],
            data_1['scale_p_10'], data_1['scale_p_20'], data_1['scale_p_30'], data_1['scale_p_40'], data_1['scale_p_50'],
            data_1['scale_p_60'], data_1['scale_p_70'], data_1['scale_p_80'], data_1['scale_p_90'],
            data_2['n_flower_o'],data_2['avg_flower_o'],data_2['n_fruit_o'],data_2['avg_fruit_o'], 
            data_2['scale_sum_r_o'], data_2['scale_max_r_o'],data_2['scale_std_r_o'],
            data_2['scale_p_10_o'], data_2['scale_p_20_o'], data_2['scale_p_30_o'], data_2['scale_p_40_o'], data_2['scale_p_50_o'],
            data_2['scale_p_60_o'], data_2['scale_p_70_o'], data_2['scale_p_80_o'], data_2['scale_p_90_o'],
            data_2['avg_flower_o_s'], data_2['avg_fruit_o_s']], axis=1, ignore_index=False
            )
            
            date1 = datetime(dates[i-1][0],dates[i-1][1],dates[i-1][2])
            date2 = datetime(dates[j-1][0],dates[j-1][1],dates[j-1][2])
            data_['time'] = (date2 - date1).days

                    
            start_date = str(dates_n[i-1])
            end_date = str(dates_n[j-1])
            # # Filter the dataframe based on the date range
            filtered_df = data_w1[(data_w1["datetime"] >= start_date) & (data_w1["datetime"] <= end_date)]
            # weather_data.append(filtered_df)
            new_df = prepare_weather_data(filtered_df, images_path, tree_n)
            combined_df = pd.concat([data_, new_df], axis=1)
            all_data.append(combined_df)

            # display(data_)
            # break
        else:
            print('not',i,j)


data_f = all_data[0]
for i in range(1,len(all_data)):
    data_f = pd.concat([data_f, all_data[i]], ignore_index=True)
# len(data_f), data_f

data_f.to_csv(csv_file, index=False)

yes:prepare_scale_factors
yes:prepare_scale_factors
yes:prepare_scale_factors
not 2 2


/tmp/ipykernel_2782652/654815068.py:173: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['preciptype'] = filtered_df['preciptype'].replace('rain', 1)
/tmp/ipykernel_2782652/654815068.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.fillna(0, inplace=True)
/tmp/ipykernel_2782652/654815068.py:208: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/in